# 19 - VoxelMorph-style en todos los sujetos vs baseline clasico

Este notebook resume el experimento deep learning instance-specific para los seis especimenes.

La red se entrena por sujeto, por lo que no se interpreta como modelo generalizable. Se interpreta como una optimizacion deep learning por pareja H&E-HSI.

## Metodo resumido

- Inicializacion: H&E ya registrada por el baseline afin/rigido.
- Fixed: HSI segmentada.
- Moving: H&E segmentada tras inicializacion.
- Entrada de red: mapas de distancia firmados de ambas mascaras.
- Salida: campo denso de deformacion.
- Perdida: MSE de mapas de distancia + Dice loss de mascaras + suavidad del campo.

El objetivo es comprobar si una CNN tipo VoxelMorph puede mejorar el ajuste deformable en nuestras muestras.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path.cwd()
METHOD_DIR = ROOT / 'Registration' / 'DeepLearning' / 'Tecnicas_registration' / '01_voxelmorph'
SCRIPT = METHOD_DIR / 'scripts' / 'run_voxelmorph_all_and_compare.py'
OUTPUT_DIR = METHOD_DIR / 'outputs' / 'outputs_voxelmorph_instance_todos'
CSV_PATH = OUTPUT_DIR / 'voxelmorph_vs_baseline_summary.csv'
print(SCRIPT)


In [ ]:
# Cambia a True si quieres relanzar los seis entrenamientos desde este notebook.
# En CPU tarda varios minutos.
RUN_ALL = False

if RUN_ALL:
    result = subprocess.run([sys.executable, str(SCRIPT)], cwd=str(ROOT), text=True, capture_output=True)
    print(result.stdout[-5000:])
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('Fallo el experimento completo de VoxelMorph-style')


In [ ]:
df = pd.read_csv(CSV_PATH)
df

In [ ]:
cols = ['subject', 'initial_dice', 'baseline_dice', 'voxelmorph_dice', 'voxelmorph_minus_baseline', 'flow_p95_px', 'flow_max_px', 'jacobian_det_negative_fraction']
df[cols]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 14))
axes[0].imshow(Image.open(OUTPUT_DIR / 'voxelmorph_vs_baseline_dice.png'))
axes[0].axis('off')
axes[0].set_title('Comparacion Dice')
axes[1].imshow(Image.open(OUTPUT_DIR / 'voxelmorph_vs_baseline_overlay_grid.png'))
axes[1].axis('off')
axes[1].set_title('Overlays baseline vs VoxelMorph-style')
plt.tight_layout()


## Lectura critica

La red mejora mucho el Dice de mascara, especialmente en SB017. Sin embargo, los campos de deformacion no son siempre suaves/topologicamente seguros: en varios sujetos aparece una fraccion de Jacobiano negativo.

Conclusion provisional: el metodo es muy util como experimento deep learning y demuestra que una red instance-specific puede ajustar contornos H&E-HSI. Antes de aceptarlo como resultado final, conviene probar una version mas conservadora aumentando la regularizacion, reduciendo el desplazamiento maximo o penalizando plegamientos.